# Bab 15: Metode Sampling dan Pemrograman Probabilistik

Notebook pendamping buku *Pengenalan Pola dan Machine Learning: Pendekatan Probabilistik dengan Python*.

Jalankan sel dari atas ke bawah (**Runtime → Run all** di Colab). Sel pertama menyiapkan folder kode dan modul data sintetis; setiap sel berikutnya adalah satu listing dari buku, dengan nomor penanda `# (1)`, `# (2)`, … yang dijelaskan di bawah listing pada buku. Angka keluaran bisa sedikit berbeda dari buku bila versi pustakanya berbeda (lihat Lampiran A).

In [ ]:
!pip install -q pymc arviz

In [ ]:
# Menyiapkan folder kode/ dan modul pendukung bab ini
import os, pathlib
BERKAS = {
"bab15/data_stunting.py": "\"\"\"Survei stunting sintetis di 30 kabupaten/kota (ilustrasi).\n\nUkuran sampel per kabupaten sangat bervariasi (12 sampai 400 balita), seperti\nsurvei yang alokasi sampelnya tidak seimbang. Prevalensi sebenarnya diambil\ndari distribusi logit-normal di sekitar 22%.\n\"\"\"\nimport numpy as np\nfrom scipy.special import expit\n\ndef buat_data(seed=7):\n    rng = np.random.default_rng(seed)\n    J = 30\n    theta = expit(rng.normal(np.log(0.22 / 0.78), 0.35, J))   # prevalensi sebenarnya\n    n = np.round(np.exp(rng.uniform(np.log(12), np.log(400), J))).astype(int)\n    y = rng.binomial(n, theta)\n    return y, n, theta\n",
"bab15/gibbs_gauss.py": "import numpy as np\n\n# Data hasil gabah yang sama dengan Bab 14 (ton/ha, sintetis).\nx = np.array([5.8, 6.4, 5.1, 6.9, 5.5, 6.2, 7.1, 5.9, 6.6, 6.0])\nN, xb = len(x), x.mean()\nmu0, lam0, a0, b0 = 5.0, 1.0, 1.0, 1.0\n\nrng = np.random.default_rng(0)\nmu, tau, sampel = xb, 1.0, []\nfor t in range(20_000):                              #<1>\n    # p(mu | tau, x): Gaussian\n    lam = (lam0 + N) * tau\n    mu = rng.normal((lam0 * mu0 + N * xb) / (lam0 + N), lam ** -0.5)\n    # p(tau | mu, x): gamma\n    a = a0 + (N + 1) / 2\n    b = b0 + 0.5 * (np.sum((x - mu) ** 2) + lam0 * (mu - mu0) ** 2)\n    tau = rng.gamma(a, 1 / b)\n    sampel.append((mu, tau))\nS = np.array(sampel[1000:])\nprint(f\"Gibbs: E[mu] = {S[:, 0].mean():.4f}, sd[mu] = {S[:, 0].std():.4f},\"\n      f\" E[tau] = {S[:, 1].mean():.4f}\")\nprint(\"eksak: E[mu] = 6.0455, sd[mu] = 0.2462, E[tau] = 1.7998   (Bab 14)\")\nprint(\"VB   : E[mu] = 6.0455, sd[mu] = 0.2247, E[tau] = 1.7998   (Bab 14)\")\nkecil = S[:, 1] < np.median(S[:, 1])                 # bentuk corong  #<2>\nprint(f\"sd[mu] bila tau < median: {S[kecil, 0].std():.3f},\"\n      f\" bila tau >= median: {S[~kecil, 0].std():.3f}\")\n",
"bab15/hierarkis_stunting.py": "import numpy as np\nimport pymc as pm\nimport arviz as az\nfrom data_stunting import buat_data\n\ny, n, theta_benar = buat_data()\nJ = len(y)\n\nwith pm.Model() as model:                                  # model hierarkis  #<1>\n    mu = pm.Normal(\"mu\", -1.3, 1.0)                        # logit rerata provinsi\n    sigma = pm.HalfNormal(\"sigma\", 0.5)                    # sebaran antarkabupaten\n    eta = pm.Normal(\"eta\", 0, 1, shape=J)                  # parameterisasi tak terpusat  #<2>\n    theta = pm.Deterministic(\"theta\", pm.math.invlogit(mu + sigma * eta))\n    pm.Binomial(\"y\", n=n, p=theta, observed=y)\n    idata = pm.sample(1000, tune=1000, chains=4, random_seed=0,\n                      progressbar=False)                   # NUTS  #<3>\n\nringkas = az.summary(idata, var_names=[\"mu\", \"sigma\"])     #<4>\nprint(\"         mean     sd ess_bulk r_hat\")\nfor v, r in ringkas.iterrows():\n    print(f\"{v:5s} {r['mean']:7.3f} {r['sd']:6.3f} {r['ess_bulk']:8.0f} {r['r_hat']:5.2f}\")\nprint(\"divergensi:\", int(idata.sample_stats[\"diverging\"].sum()))\nth = idata.posterior[\"theta\"].mean((\"chain\", \"draw\")).values\nmentah = y / n\nkecil = n < 50\nprint(\"galat absolut rata-rata   semua   n < 50   n >= 50\")\nfor nama, est in [(\"y/n saja \", mentah), (\"gabungan \", np.full(J, y.sum() / n.sum())),\n                  (\"hierarkis\", th)]:                      # tiga strategi  #<5>\n    e = np.abs(est - theta_benar)\n    print(f\"  {nama}               {e.mean():.4f}   {e[kecil].mean():.4f}   {e[~kecil].mean():.4f}\")\nfor j in np.argsort(n)[[0, 1, -1]]:\n    print(f\"  kab. #{j:2d}: n = {n[j]:3d}, y/n = {mentah[j]:.3f},\"\n          f\" hierarkis = {th[j]:.3f}, sebenarnya = {theta_benar[j]:.3f}\")\n",
"bab15/metropolis.py": "import numpy as np\nfrom scipy.special import expit\n\n# 12 kabupaten (sintetis): akses sanitasi layak (dibakukan) dan kasus stunting.\nrng = np.random.default_rng(3)\nx = rng.normal(0, 1, 12); n = rng.integers(60, 200, 12)\ny = rng.binomial(n, expit(-1.3 - 0.4 * x))\n\ndef log_post(th):                                   # logit p = a + b x, prior N(0, 2^2)\n    a, b = th; eta = a + b * x\n    return np.sum(y * eta - n * np.logaddexp(0, eta)) - (a ** 2 + b ** 2) / 8\n\ndef metropolis(langkah, T=5000, seed=0):            #<1>\n    r = np.random.default_rng(seed)\n    th, lp, jejak, terima = np.zeros(2), log_post(np.zeros(2)), [], 0\n    for _ in range(T):\n        usul = th + langkah * r.normal(size=2)      # usulan simetris\n        lp_u = log_post(usul)\n        if np.log(r.uniform()) < lp_u - lp:         # aturan penerimaan  #<2>\n            th, lp, terima = usul, lp_u, terima + 1\n        jejak.append(th)\n    return np.array(jejak), terima / T\n\ndef ess(z):                                         # ukuran sampel efektif  #<3>\n    z = z - z.mean(); n = len(z)\n    rho = np.correlate(z, z, \"full\")[n - 1:] / (z.var() * n)\n    k = np.argmax(rho < 0.05)                       # potong saat autokorelasi kecil\n    return n / (1 + 2 * rho[1:k].sum())\n\nfor langkah in [0.005, 0.08, 1.0]:\n    J, acc = metropolis(langkah)\n    b = J[1000:, 1]                                 # buang 1000 sampel awal  #<4>\n    print(f\"langkah {langkah:5.3f}: diterima {acc:5.1%}, E[b] = {b.mean():+.3f},\"\n          f\" sd[b] = {b.std():.3f}, ESS {ess(b):4.0f}\")\n",
"bab15/rejection_importance.py": "import numpy as np\nfrom scipy.stats import norm, binom, beta\nfrom scipy.special import logit\n\n# Survei di satu kabupaten (sintetis): 41 dari 150 balita stunting.\ny, n = 41, 150\nlog_prior = lambda t: norm.logpdf(logit(t), logit(0.21), 0.5) - np.log(t * (1 - t))\nlog_target = lambda t: log_prior(t) + binom.logpmf(y, n, t)    # tak ternormalisasi  #<1>\n\ng = np.linspace(1e-4, 1 - 1e-4, 20001)                        # rujukan: grid\nw = np.exp(log_target(g) - log_target(g).max()); w /= w.sum()\nprint(f\"grid       : E[theta] = {np.sum(g * w):.4f}, p(>0.30) = {w[g > 0.3].sum():.4f}\")\n\nrng = np.random.default_rng(0)\nM = np.exp(log_target(g).max())                              # batas atas, usulan seragam  #<2>\nt = rng.uniform(0, 1, 200_000)\nu = rng.uniform(0, M, t.size)\ns = t[u < np.exp(log_target(t))]\nprint(f\"rejection  : E[theta] = {s.mean():.4f}, p(>0.30) = {np.mean(s > 0.3):.4f},\"\n      f\" terima {s.size / t.size:.1%}\")\n\nq = beta(8, 32)                                               # usulan importance  #<3>\nt = q.rvs(20_000, random_state=1)\nlw = log_target(t) - q.logpdf(t); w = np.exp(lw - lw.max()); w /= w.sum()\ness = 1 / np.sum(w ** 2)                                      # ukuran sampel efektif  #<4>\nprint(f\"importance : E[theta] = {np.sum(w * t):.4f}, p(>0.30) = {w[t > 0.3].sum():.4f},\"\n      f\" ESS {ess:.0f} dari {t.size}\")\n"
}
akar = pathlib.Path('pola-ml/kode').resolve()
for nama, isi in BERKAS.items():
    p = akar / nama; p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(isi, encoding='utf-8')
os.chdir(akar / 'bab15')
print('folder kerja:', os.getcwd())

## Listing 15.1: Posterior prevalensi stunting dengan rejection dan importance sampling

Berkas: `kode/bab15/rejection_importance.py`

In [ ]:
import numpy as np
from scipy.stats import norm, binom, beta
from scipy.special import logit

# Survei di satu kabupaten (sintetis): 41 dari 150 balita stunting.
y, n = 41, 150
log_prior = lambda t: norm.logpdf(logit(t), logit(0.21), 0.5) - np.log(t * (1 - t))
log_target = lambda t: log_prior(t) + binom.logpmf(y, n, t)    # tak ternormalisasi  # (1)

g = np.linspace(1e-4, 1 - 1e-4, 20001)                        # rujukan: grid
w = np.exp(log_target(g) - log_target(g).max()); w /= w.sum()
print(f"grid       : E[theta] = {np.sum(g * w):.4f}, p(>0.30) = {w[g > 0.3].sum():.4f}")

rng = np.random.default_rng(0)
M = np.exp(log_target(g).max())                              # batas atas, usulan seragam  # (2)
t = rng.uniform(0, 1, 200_000)
u = rng.uniform(0, M, t.size)
s = t[u < np.exp(log_target(t))]
print(f"rejection  : E[theta] = {s.mean():.4f}, p(>0.30) = {np.mean(s > 0.3):.4f},"
      f" terima {s.size / t.size:.1%}")

q = beta(8, 32)                                               # usulan importance  # (3)
t = q.rvs(20_000, random_state=1)
lw = log_target(t) - q.logpdf(t); w = np.exp(lw - lw.max()); w /= w.sum()
ess = 1 / np.sum(w ** 2)                                      # ukuran sampel efektif  # (4)
print(f"importance : E[theta] = {np.sum(w * t):.4f}, p(>0.30) = {w[t > 0.3].sum():.4f},"
      f" ESS {ess:.0f} dari {t.size}")

## Listing 15.2: Metropolis dari nol untuk regresi logistik stunting–sanitasi

Berkas: `kode/bab15/metropolis.py`

In [ ]:
import numpy as np
from scipy.special import expit

# 12 kabupaten (sintetis): akses sanitasi layak (dibakukan) dan kasus stunting.
rng = np.random.default_rng(3)
x = rng.normal(0, 1, 12); n = rng.integers(60, 200, 12)
y = rng.binomial(n, expit(-1.3 - 0.4 * x))

def log_post(th):                                   # logit p = a + b x, prior N(0, 2^2)
    a, b = th; eta = a + b * x
    return np.sum(y * eta - n * np.logaddexp(0, eta)) - (a ** 2 + b ** 2) / 8

def metropolis(langkah, T=5000, seed=0):            # (1)
    r = np.random.default_rng(seed)
    th, lp, jejak, terima = np.zeros(2), log_post(np.zeros(2)), [], 0
    for _ in range(T):
        usul = th + langkah * r.normal(size=2)      # usulan simetris
        lp_u = log_post(usul)
        if np.log(r.uniform()) < lp_u - lp:         # aturan penerimaan  # (2)
            th, lp, terima = usul, lp_u, terima + 1
        jejak.append(th)
    return np.array(jejak), terima / T

def ess(z):                                         # ukuran sampel efektif  # (3)
    z = z - z.mean(); n = len(z)
    rho = np.correlate(z, z, "full")[n - 1:] / (z.var() * n)
    k = np.argmax(rho < 0.05)                       # potong saat autokorelasi kecil
    return n / (1 + 2 * rho[1:k].sum())

for langkah in [0.005, 0.08, 1.0]:
    J, acc = metropolis(langkah)
    b = J[1000:, 1]                                 # buang 1000 sampel awal  # (4)
    print(f"langkah {langkah:5.3f}: diterima {acc:5.1%}, E[b] = {b.mean():+.3f},"
          f" sd[b] = {b.std():.3f}, ESS {ess(b):4.0f}")

## Listing 15.3: Gibbs sampling untuk rerata dan presisi Gaussian

Berkas: `kode/bab15/gibbs_gauss.py`

In [ ]:
import numpy as np

# Data hasil gabah yang sama dengan Bab 14 (ton/ha, sintetis).
x = np.array([5.8, 6.4, 5.1, 6.9, 5.5, 6.2, 7.1, 5.9, 6.6, 6.0])
N, xb = len(x), x.mean()
mu0, lam0, a0, b0 = 5.0, 1.0, 1.0, 1.0

rng = np.random.default_rng(0)
mu, tau, sampel = xb, 1.0, []
for t in range(20_000):                              # (1)
    # p(mu | tau, x): Gaussian
    lam = (lam0 + N) * tau
    mu = rng.normal((lam0 * mu0 + N * xb) / (lam0 + N), lam ** -0.5)
    # p(tau | mu, x): gamma
    a = a0 + (N + 1) / 2
    b = b0 + 0.5 * (np.sum((x - mu) ** 2) + lam0 * (mu - mu0) ** 2)
    tau = rng.gamma(a, 1 / b)
    sampel.append((mu, tau))
S = np.array(sampel[1000:])
print(f"Gibbs: E[mu] = {S[:, 0].mean():.4f}, sd[mu] = {S[:, 0].std():.4f},"
      f" E[tau] = {S[:, 1].mean():.4f}")
print("eksak: E[mu] = 6.0455, sd[mu] = 0.2462, E[tau] = 1.7998   (Bab 14)")
print("VB   : E[mu] = 6.0455, sd[mu] = 0.2247, E[tau] = 1.7998   (Bab 14)")
kecil = S[:, 1] < np.median(S[:, 1])                 # bentuk corong  # (2)
print(f"sd[mu] bila tau < median: {S[kecil, 0].std():.3f},"
      f" bila tau >= median: {S[~kecil, 0].std():.3f}")

## Listing 15.4: Model hierarkis prevalensi stunting dengan PyMC

Berkas: `kode/bab15/hierarkis_stunting.py`

In [ ]:
import numpy as np
import pymc as pm
import arviz as az
from data_stunting import buat_data

y, n, theta_benar = buat_data()
J = len(y)

with pm.Model() as model:                                  # model hierarkis  # (1)
    mu = pm.Normal("mu", -1.3, 1.0)                        # logit rerata provinsi
    sigma = pm.HalfNormal("sigma", 0.5)                    # sebaran antarkabupaten
    eta = pm.Normal("eta", 0, 1, shape=J)                  # parameterisasi tak terpusat  # (2)
    theta = pm.Deterministic("theta", pm.math.invlogit(mu + sigma * eta))
    pm.Binomial("y", n=n, p=theta, observed=y)
    idata = pm.sample(1000, tune=1000, chains=4, random_seed=0,
                      progressbar=False)                   # NUTS  # (3)

ringkas = az.summary(idata, var_names=["mu", "sigma"])     # (4)
print("         mean     sd ess_bulk r_hat")
for v, r in ringkas.iterrows():
    print(f"{v:5s} {r['mean']:7.3f} {r['sd']:6.3f} {r['ess_bulk']:8.0f} {r['r_hat']:5.2f}")
print("divergensi:", int(idata.sample_stats["diverging"].sum()))
th = idata.posterior["theta"].mean(("chain", "draw")).values
mentah = y / n
kecil = n < 50
print("galat absolut rata-rata   semua   n < 50   n >= 50")
for nama, est in [("y/n saja ", mentah), ("gabungan ", np.full(J, y.sum() / n.sum())),
                  ("hierarkis", th)]:                      # tiga strategi  # (5)
    e = np.abs(est - theta_benar)
    print(f"  {nama}               {e.mean():.4f}   {e[kecil].mean():.4f}   {e[~kecil].mean():.4f}")
for j in np.argsort(n)[[0, 1, -1]]:
    print(f"  kab. #{j:2d}: n = {n[j]:3d}, y/n = {mentah[j]:.3f},"
          f" hierarkis = {th[j]:.3f}, sebenarnya = {theta_benar[j]:.3f}")